# Programation Par Contraite - Covering Arrays avec contraintes semantiques

In [1]:
from ortools.sat.python import cp_model
import pandas as pd
import itertools
import random
from abc import ABC, abstractmethod
import functools
import math
from typing import Any, TypeAlias, Callable
from time import perf_counter

## Qu'est ce qu'un Covering Array ?

> Les Covering Arrays (CA) sont des matrices $N \times k$ où chaque colonne représente un paramètre (facteur) avec $v$ valeurs, telles que toute combinaison de $t$ colonnes contienne tous les $t$-uplets possibles au moins une fois.

Dit comme cela, la définition peut sembler un peu abstraite, alors voyons un exemple extrêmement simple.

Imaginons un formulaire où nous demandons la majorité du participant (Majeur ou non), son statut professionnel (Employé ou non) et sa nationalité (Française ou non). Nous avons donc 3 paramètres ($k = 3$), et chacun peut prendre exactement 2 valeurs ($v = 2$).

En théorie, il y a $2 \times 2 \times 2 = 2^3 = v^k = 8$ combinaisons possibles si on voulait tout tester. L'idée du Covering Array est de ne pas tester toutes les combinaisons globales, mais de garantir la couverture sur un groupe de $t$ colonnes. Si on prend une force $t = 2$ (*pairwise testing*), on obtient une matrice où chaque paire de colonnes contient toutes les paires de valeurs possibles au moins une fois.

$$
\text{CA} = \begin{bmatrix}
\text{Majeur} & \text{Employé} & \text{Français} \\
0 & 0 & 1 \\
1 & 0 & 0 \\
0 & 1 & 0 \\
1 & 1 & 1 
\end{bmatrix}
$$

Ici, on peut voir que si on isole deux colonnes au choix (par exemple *Majeur* et *Français*), on retrouve bien les 4 paires possibles : $(0, 0)$, $(0, 1)$, $(1, 0)$ et $(1, 1)$. 

Tous les cas globaux n'ont pas été testés (seulement 4 lignes sur 8), mais cette méthode permet de réduire grandement le nombre de tests tout en gardant une excellente capacité à détecter les bugs d'interactions.

## Contraintes Sémentiques

Vous avez peut-être relevé un problème logique dans l'exemple précédent. En effet, la troisième ligne propose un profil **Mineur (0) et Employé (1)**, ce qui est interdit par la loi (plus ou moins). 

Pour résoudre ce problème, on ajoute des **contraintes sémantiques** exprimées en logique propositionnelle : ici, $\neg(\text{Mineur} \land \text{Employé})$. 

Exclure des combinaisons rend le problème plus difficile pour le solveur. Comme certaines lignes "partagées" deviennent interdites, le solveur doit réorganiser la matrice et est parfois obligé d'**augmenter le nombre de lignes $N$** pour réussir à couvrir le reste des paires légitimes.

Avec notre contrainte, le nombre minimal de lignes passe de $N=4$ à $N=5$ :

$$
\text{CA}_{\text{contraint}} = \begin{bmatrix}
\text{Majeur} & \text{Employé} & \text{Français} \\
1 & 1 & 0 \\
1 & 0 & 1 \\
0 & 0 & 0 \\
0 & 0 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$

On remarque qu'aucune ligne ne contient la combinaison interdite $(0, 1)$ sur les colonnes Majeur/Employé, mais que toutes les autres paires du système restent testées.

## Sujet

Dans ce projet, nous allons nous intéresser a la génération de Covering Array avec contraintes sémentiques et une force $t \ge 3$.

In [2]:
Parameter: TypeAlias = str
Value: TypeAlias = Any
Values: TypeAlias = list[Value]
Parameters: TypeAlias = dict[Parameter, Values]
ParameterMap: TypeAlias = dict[Parameter, dict[Value, int]]
Constraint: TypeAlias = Callable[[cp_model.CpModel, dict[Parameter, cp_model.IntVar], ParameterMap], None]

In [3]:
def lower_bound(parameters: Parameters, t: int) -> int:
    """
    Computes a combinatorial lower bound for the size of a t-wise covering array.

    For each combination of `t` parameters, the function computes the product of the
    number of values in their domains. The maximum of these products is returned as a
    valid lower bound on the number of rows required to cover all t-wise interactions.

    This bound is only valid when no sementic constraints are applied.

    @param parameters:
        Dictionary mapping parameter names to a list of possible values.

    @param t:
        Strength of interaction (size of combinations to cover).

    @return:
        Integer lower bound on the number of rows required for a t-wise covering array.def
    """
    values = list(parameters.values())

    best = 0

    for subset in itertools.combinations(values, t):
        prod = math.prod(len(v) for v in subset)
        best = max(best, prod)

    return best

In [13]:
class CoveringArray:
    """
    Represents a t-wise covering array solution.

    Attributes:
        matrix (list[list[int]]):
            Internal representation of the covering array.
            Each row contains integer indices corresponding to parameter values.

        parameters (Parameters):
            Dictionary mapping parameter names to their list of possible values.

        t (int):
            T force of the covering array
    """

    def __init__(self, matrix: list[list[int]], parameters: Parameters, t: int) -> None:
        """
        Constructs a CoveringArray instance.

        @param matrix:
            2D list representing the covering array. Each row is a test case,
            and each value is an index into the corresponding parameter domain.

        @param parameters:
            Dictionary mapping parameter names to their possible values.
        """
        self.matrix = matrix
        self.parameters = parameters
        self.t = t

    def get_matrix(self) -> list[list[int]]:
        """
        Returns the internal matrix representation of the covering array.

        @return:
            A 2D list of integers representing encoded test cases.
        """
        return self.matrix

    def to_human_readable(self) -> list[dict[Parameter, Value]]:
        """
        Converts the internal integer-encoded matrix into a human-readable form.

        Each row is transformed into a dictionary mapping parameter names
        to their corresponding actual values.

        @return:
            A list of dictionaries, each representing a decoded test case.
        """
        return [
            {
                parameter_name: self.parameters[parameter_name][row[column_idx]]
                for column_idx, parameter_name in enumerate(self.parameters.keys())
            }
            for row in self.matrix
        ]

    def to_data_frame(self) -> pd.DataFrame:
        """
        Converts the covering array into a readable deta frame.

        @return:
            The pandas DataFrame.
        """
        return pd.DataFrame(self.to_human_readable())

    def is_valid(self) -> (bool, list[tuple]):
        """
        Verifies whether a covering array satisfies full t-wise coverage.
        
        @return:
            (True, []) if all t-wise interactions are covered, (False, list of missing interactions) otherwise.
        """
        rows = self.to_human_readable()
        parameters = self.parameters
        t = self.t
    
        parameter_names = list(parameters.keys())
    
        valid = True
        missings = []
    
        for subset in itertools.combinations(parameter_names, t):
            domains = [parameters[p] for p in subset]
    
            required = set(itertools.product(*domains))
    
            observed = {
                tuple(row[p] for p in subset)
                for row in rows
            }
    
            missing = required - observed
    
            if missing:
                valid = False
    
                for interaction in missing:
                    missings.append((subset, interaction))
    
        return (valid, missings)

In [5]:
class CoveringArrayGenerator:
    """
    Abstract base class for all covering array generators.

    This ensures a unified interface for:
    - CP-SAT based generation
    - IPOG heuristic generation
    - AETG stochastic generation
    """

    def __init__(self, parameters: Parameters, t: int):
        """
        @param parameters:
            Dictionary mapping parameter names to possible values.

        @param t:
            Strength of interaction (t-wise coverage).
        """
        self.parameters = parameters
        self.t = t
        self.parameter_map: ParameterMap = {
            parameter: {value: index for index, value in enumerate(values)}
            for parameter, values in parameters.items()
        }

    @abstractmethod
    def generate(self) -> CoveringArray:
        """
        Generates a t-wise covering array.

        @return:
            CoveringArray instance.
        """
        pass

In [14]:
class CPSATGenerator(CoveringArrayGenerator):
    """
    CP-SAT based covering array generator.

    This class builds a constraint programming model (using OR-Tools CP-SAT)
    to compute a t-wise covering array under optional user-defined constraints.

    Attributes:
        parameters (Parameters):
            Dictionary mapping parameter names to their possible values.

        t (int):
            Strength of interaction (t-wise coverage).

        constraints (list[Constraint]):
            User-defined constraints applied to each row of the model.

        parameter_map (ParameterMap):
            Mapping from parameter values to integer indices for CP-SAT encoding.
    """
    
    def __init__(self, parameters: Parameters, t: int) -> None:
        """
        Initializes the CP-SAT covering array builder.

        @param parameters:
            Dictionary mapping parameter names to their possible values.

        @param t:
            Strength of interaction (t-wise coverage requirement).

        @raises AssertionError:
            If number of parameters is smaller than t.
        """
        super().__init__(parameters, t)

        assert len(parameters) >= self.t, f"We can't have less parameters than t force, got {len(parameters)} parameters but {self.t} t force"
    
        self.constraints: list[Constraint] = []

    def add_constraint(self, constraint: Constraint) -> "CoveringArrayBuilder":
        """
        Adds a user-defined constraint to the CP-SAT model.

        Constraints are applied to every row in the covering array.

        @param constraint:
            A function of the form:
            (model, row_variables, parameter_map) -> None

        @return:
            self (for method chaining).
        """
        self.constraints.append(constraint)
        return self

    def _build_model(self) -> cp_model.CpModel:
        """
        Constructs the CP-SAT model for the covering array problem.

        The model includes:
        - Variable encoding of all possible rows
        - Row activation constraints
        - User-defined constraints
        - Value coverage constraints
        - t-wise interaction coverage constraints
        - Objective: minimize number of used rows

        @return:
            A tuple containing:
            - CP-SAT model
            - variable matrix x
            - row usage indicators
        """
        model = cp_model.CpModel()

        v = [len(values) for values in self.parameters.values()]
        N_max: int = functools.reduce(lambda x, y: x * y, v)

        # Gives the solver a matrix of all possible values for each variables
        x: list[dict[Parameter, cp_model.IntVar]] = [
            {
                parameter: model.new_int_var(0, len(values) - 1, f"x_{index}_{parameter}")
                for parameter, values in self.parameters.items()
            }
            for index in range(N_max)
        ]

        # Forces the solver to use rows in order
        row_is_used = [model.new_bool_var(f"row_used_{index}") for index in range(N_max)]
        for index in range(N_max - 1):
            model.add_implication(row_is_used[index + 1], row_is_used[index])
    
        # Applies the different user defined constraints to the model
        for row in x:
            for constraint in self.constraints:
                constraint(model, row, self.parameter_map)

        # Forces each variable possible value to appear at least once
        for parameter, values in self.parameters.items():
            for value in values:
                matches = []
                
                for index, row in enumerate(x):
                    b = model.new_bool_var(f"has_{parameter}_{value}_row{index}")
                    model.add(row[parameter] == self.parameter_map[parameter][value]).only_enforce_if(b)
                    model.add(row[parameter] != self.parameter_map[parameter][value]).only_enforce_if(b.Not())
        
                    matches.append(b)

                model.add_bool_or(matches)

        # Generate all t-wise interactions
        for parameter_subset in itertools.combinations(self.parameters.keys(), self.t):
            domains = [self.parameters[parameter] for parameter in parameter_subset]
    
            for values_tuple in itertools.product(*domains):
                # If the interaction is impossible due to user defined constraits we don't include it
                if not self._is_interaction_feasible(parameter_subset, values_tuple):
                    continue
    
                # Create "row matches interaction" booleans
                matching_rows = []
    
                for row_index, row in enumerate(x):
                    row_matches = model.new_bool_var(
                        f"match_{row_index}_{'_'.join(str(parameter_subset))}_{'_'.join(map(str, values_tuple))}"
                    )
    
                    conditions = []
    
                    # row[param] == wanted_value
                    for parameter, value in zip(parameter_subset, values_tuple):
                        b = model.new_bool_var(f"eq_{row_index}_{parameter}_{value}")
                        model.add(row[parameter] == self.parameter_map[parameter][value]).only_enforce_if(b)
                        model.add(row[parameter] != self.parameter_map[parameter][value]).only_enforce_if(b.Not())
    
                        conditions.append(b)
    
                    # row_matches <=> all conditions true
                    model.add_bool_and(conditions).only_enforce_if(row_matches)

                    # if row_matches then row must be used
                    model.add_implication(row_matches, row_is_used[row_index])
    
                    matching_rows.append(row_matches)
    
                # Coverage constraint: at least one row matches this interaction
                model.add_bool_or(matching_rows)
    
        # Objective: minimize N the number of rows used
        model.minimize(sum(row_is_used))

        return model, x, row_is_used

    def _is_interaction_feasible(self, parameter_subset, values_tuple) -> bool:
        """
        Checks whether a given t-wise interaction is feasible under user constraints.

        This avoids generating constraints for impossible combinations.

        @param parameter_subset:
            Subset of parameter names.

        @param values_tuple:
            Corresponding values for the subset.

        @return:
            True if the interaction is satisfiable, False otherwise.
        """
        test_model = cp_model.CpModel()

        # Create one fake row
        row = {
            parameter: test_model.new_int_var(0, len(self.parameters[parameter]) - 1, f"{parameter}")
            for parameter in self.parameters
        }
    
        # Apply all user constraints
        for constraint in self.constraints:
            constraint(test_model, row, self.parameter_map)
    
        # Force the tested interaction
        for parameter, value in zip(parameter_subset, values_tuple):
            test_model.add(row[parameter] == self.parameter_map[parameter][value])
    
        # Check satisfiable
        solver = cp_model.CpSolver()
    
        status = solver.solve(test_model)
    
        return status in (cp_model.FEASIBLE, cp_model.OPTIMAL)

    def generate(self, max_time_seconds: float = 180.0) -> CoveringArray:
        """
        Solves the CP-SAT model and returns a covering array.

        @param max_time_seconds:
            Maximum solving time allowed for CP-SAT. Defaults to 10 seconds.

        @return:
            A CoveringArray instance containing the solution.

        @raises RuntimeError:
            If no feasible covering array is found.
        """
        model, x, row_is_used = self._build_model()
        
        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = max_time_seconds
    
        status = solver.solve(model)
    
        if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            raise RuntimeError("No covering array found")
    
        matrix = [
            [solver.value(row_variables[parameter]) for parameter in self.parameters]
            for row_index, row_variables in enumerate(x)
            if solver.value(row_is_used[row_index])
        ]
    
        return CoveringArray(matrix, self.parameters, self.t)

In [16]:
class IPOGGenerator(CoveringArrayGenerator):
    """
    IPOG (In-Parameter-Order Generalized) covering array generator.

    This is a greedy deterministic heuristic that incrementally builds a
    t-wise covering array by:
    - initializing with all combinations of the first t parameters
    - extending the test set parameter by parameter
    - using horizontal and vertical extension phases

    Attributes:
        parameters (Parameters):
            Dictionary mapping parameter names to their possible values.

        t (int):
            Strength of interaction (t-wise coverage).

        parameter_map (ParameterMap):
            Mapping from parameter values to integer indices for CP-SAT encoding.
    """

    def __init__(self, parameters: Parameter, t: int) -> None:
        """
        Initializes the IPOG generator.
    
        @param parameters:
            Dictionary mapping parameter names to their possible values.

        @param t:
            Interaction strength (t-wise coverage).
        """
        super().__init__(parameters, t)

    def generate(self) -> CoveringArray:
        """
        Generates a t-wise covering array using the IPOG algorithm.

        @return:
            CoveringArray instance representing the generated test suite.
        """
        t = self.t
        parameter_names = list(self.parameters.keys())

        # 1. Initialize test set with all combinations of first t parameters
        test_set: list[dict[Parameter, Value]] = []

        first_parameters = parameter_names[:t]

        for values in itertools.product(*(self.parameters[p] for p in first_parameters)):
            row = {
                parameter: value
                for parameter, value in zip(first_parameters, values)
            }
            test_set.append(row)

        # 2. Incrementally add remaining parameters
        for i in range(t, len(parameter_names)):
            parameter_i = parameter_names[i]

            # Build all t-wise interactions involving parameter_i
            interactions = set()

            for subset in itertools.combinations(parameter_names[:i], t - 1):
                interaction_parameters = list(subset) + [parameter_i]

                for values in itertools.product(*(self.parameters[p] for p in interaction_parameters)):
                    interaction = tuple(sorted(zip(interaction_parameters, values)))
                    interactions.add(interaction)

            # Horizontal extension
            for row in test_set:
                best_value = None
                best_score = -1

                for value in self.parameters[parameter_i]:
                    candidate = row.copy()
                    candidate[parameter_i] = value

                    score = 0

                    for interaction in interactions:
                        if all(candidate.get(p) == v for p, v in interaction):
                            score += 1

                    if score > best_score:
                        best_score = score
                        best_value = value

                row[parameter_i] = best_value

                # remove covered interactions
                covered = {
                    interaction
                    for interaction in interactions
                    if all(row.get(p) == v for p, v in interaction)
                }

                interactions -= covered

            # Vertical extension
            while interactions:
                interaction = interactions.pop()

                new_row: dict[Parameter, Value] = {}

                # fill interaction values
                for parameter, value in interaction:
                    new_row[parameter] = value

                # fill remaining parameters arbitrarily
                for parameter in parameter_names[:i + 1]:
                    if parameter not in new_row:
                        new_row[parameter] = self.parameters[parameter][0]

                test_set.append(new_row)

                # remove newly covered interactions
                covered = {
                    it
                    for it in interactions
                    if all(new_row.get(p) == v for p, v in it)
                }

                interactions -= covered

        matrix = [
            [
                self.parameter_map[parameter][row[parameter]]
                for parameter in self.parameters
            ]
            for row in test_set
        ]

        return CoveringArray(matrix, self.parameters, self.t)

In [17]:
class AETGGenerator(CoveringArrayGenerator):
    """
    AETG (Automatic Efficient Test Generator) covering array generator.

    This generator uses a randomized greedy heuristic to construct a t-wise
    covering array. At each iteration, multiple random candidate rows are
    generated and scored according to the number of uncovered interactions
    they cover.

    The best candidate is selected and added to the test suite until all
    t-wise interactions are covered.

    Multiple independent suites can be generated, with the smallest one
    retained as the final result.

    Attributes:
        parameters (Parameters):
            Dictionary mapping parameter names to possible values.

        t (int):
            Interaction strength (t-wise coverage).
    
        parameter_map (ParameterMap):
            Mapping from parameter values to integer indices.

        suites (int):
            Number of independent full-suite generations.

        candidates (int):
            Number of random candidate rows evaluated per iteration.
    """

    def __init__(self, parameters: Parameters, t: int, suites: int = 100, candidates: int = 50) -> None:
        """
        Initializes the AETG generator.

        @param parameters:
            Dictionary mapping parameter names to their possible values.

        @param t:
            Interaction strength (t-wise coverage).

        @param suites:
            Number of independent suites to generate.

        @param candidates:
            Number of candidate rows evaluated per iteration.
        """
        super().__init__(parameters, t)

        self.suites = suites
        self.candidates = candidates

    def generate(self) -> CoveringArray:
        """
        Generates a t-wise covering array using the AETG heuristic.

        The algorithm:
        - generates all required t-wise interactions
        - repeatedly samples random candidate rows
        - selects the candidate covering the most remaining interactions
        - removes covered interactions
        - repeats until full coverage is achieved

        Multiple suites are generated, and the smallest one is returned.

        @return:
            A CoveringArray instance representing the generated test suite.
        """   
        t = self.t
        param_names = list(self.parameters.keys())
    
        # Generate all t-wise interactions
        def generate_interactions():
            interactions = set()
    
            for subset in itertools.combinations(param_names, t):
                domains = [self.parameters[p] for p in subset]
    
                for values in itertools.product(*domains):
                    interactions.add(tuple(zip(subset, values)))
    
            return interactions
    
        # Check coverage
        def covers(row, interaction):
            return all(row.get(p) == v for p, v in interaction)
    
        # Score a candidate row
        def score(row, remaining):
            return sum(1 for it in remaining if covers(row, it))
    
        # Build best suite across multiple runs (like original C++ loop)
        best_suite = None
        best_size = float("inf")
    
        all_interactions = generate_interactions()
    
        for _ in range(self.suites):
            remaining = set(all_interactions)
            suite = []
    
            while remaining:
                best_candidate = None
                best_score = -1
    
                # Generate candidate solutions
                for _ in range(self.candidates):
                    candidate = {
                        p: random.choice(self.parameters[p])
                        for p in param_names
                    }
    
                    s = score(candidate, remaining)
    
                    if s > best_score:
                        best_score = s
                        best_candidate = candidate
    
                suite.append(best_candidate)
    
                # Remove covered interactions
                remaining = {
                    it for it in remaining
                    if not covers(best_candidate, it)
                }
    
            if len(suite) < best_size:
                best_suite = suite
                best_size = len(suite)

        matrix = [
            [
                self.parameter_map[parameter][row[parameter]]
                for parameter in self.parameters
            ]
            for row in best_suite
        ]
    
        return CoveringArray(matrix, self.parameters, self.t)

## Example

Reprenons la situation initial et complexifions la un peu.

Au lieu d'avoir juste Majeur ou Mineur, nous avons 4 groupes d'âges d'enfant à senior. Les personnes peuvent avoir un permis et le droit de conduire. Et pour finir, au lieu d'avoir Français ou Etranger, nous avons maintenant les 4 continents.

Puis ajoutons quelques contraintes :
- Un enfant ne peut pas travailler
- Un enfant ne peut pas avoir de permis
- Un senior ne peut pas travailler
- Il faut un permis pour pouvoir conduire (mais on peut ne pas pouvoir conduire même avec un permis)

Trouvons alors un Covering Array pour cette situation avec une t force de 3. 

In [9]:
example_param = {
    "AgeGroup": ["Child", "Teen", "Adult", "Senior"],
    "License": ["Yes", "No"],
    "Employment": ["Employed", "Unemployed"],
    "Driving": ["Allowed", "NotAllowed"],
    "Continent": ["Eurasian", "African", "American", "Oceanian"],
}

def child_cannot_have_licence_or_work_constraint(model: cp_model.CpModel, row: dict[Parameter, cp_model.IntVar], parameter_map: ParameterMap) -> None:
    is_child = model.new_bool_var("is_child")
    model.add(row["AgeGroup"] == parameter_map["AgeGroup"]["Child"]).only_enforce_if(is_child)
    model.add(row["AgeGroup"] != parameter_map["AgeGroup"]["Child"]).only_enforce_if(is_child.Not())
    
    model.add(row["License"] != parameter_map["License"]["Yes"]).only_enforce_if(is_child)
    model.add(row["Employment"] != parameter_map["Employment"]["Employed"]).only_enforce_if(is_child)

def seniors_cannot_work_constraint(model: cp_model.CpModel, row: dict[Parameter, cp_model.IntVar], parameter_map: ParameterMap) -> None:
    is_senior = model.new_bool_var("is_senior")
    model.add(row["AgeGroup"] == parameter_map["AgeGroup"]["Senior"]).only_enforce_if(is_senior)
    model.add(row["AgeGroup"] != parameter_map["AgeGroup"]["Senior"]).only_enforce_if(is_senior.Not())
    
    model.add(row["Employment"] != parameter_map["Employment"]["Employed"]).only_enforce_if(is_senior)

def need_licence_to_drive_constraint(model: cp_model.CpModel, row: dict[Parameter, cp_model.IntVar], parameter_map: ParameterMap) -> None:
    no_license = model.new_bool_var("no_license")
    model.add(row["License"] == parameter_map["License"]["No"]).only_enforce_if(no_license)
    model.add(row["License"] != parameter_map["License"]["No"]).only_enforce_if(no_license.Not())

    model.Add(row["Driving"] != parameter_map["Driving"]["Allowed"]).only_enforce_if(no_license)

CA_builder = CPSATGenerator(example_param, t=3)
CA_builder.add_constraint(child_cannot_have_licence_or_work_constraint)
CA_builder.add_constraint(seniors_cannot_work_constraint)
CA_builder.add_constraint(need_licence_to_drive_constraint)
CA = CA_builder.generate()
CA.to_data_frame()

,AgeGroup,License,Employment,Driving,Continent
0,Senior,Yes,Unemployed,NotAllowed,Oceanian
1,Senior,No,Unemployed,NotAllowed,American
2,Senior,No,Unemployed,NotAllowed,African
3,Senior,No,Unemployed,NotAllowed,Eurasian
4,Senior,Yes,Unemployed,Allowed,Oceanian
5,Senior,Yes,Unemployed,Allowed,American
6,Senior,Yes,Unemployed,Allowed,African
7,Senior,Yes,Unemployed,Allowed,Eurasian
8,Adult,No,Employed,NotAllowed,Oceanian
9,Teen,Yes,Employed,NotAllowed,American


In [10]:
def gen_benchmark_parameters(k: int, v: int) -> dict[int, list[int]]:
    return {
        ki: [vi for vi in range(v)]
        for ki in range(k)
    }

In [11]:
p = gen_benchmark_parameters(5, 2)

In [18]:
CA_generator_CPSAT = CPSATGenerator(p, t=3)
CA1 = CA_generator_CPSAT.generate()
print(CA1.is_valid())
CA1.to_data_frame()

(True, [])


,0,1,2,3,4
0,0,0,0,0,0
1,0,1,1,0,1
2,0,1,0,1,1
3,1,1,0,0,0
4,1,0,1,0,0
5,1,0,0,1,0
6,1,1,1,1,1
7,0,1,1,1,0
8,0,0,1,1,1
9,1,0,0,0,1


In [20]:
CA_generator_IPOG = IPOGGenerator(p, t=3)
CA2 = CA_generator_IPOG.generate()
print(CA2.is_valid())
CA2.to_data_frame()

(True, [])


,0,1,2,3,4
0,0,0,0,0,0
1,0,0,1,1,1
2,0,1,0,1,0
3,0,1,1,0,1
4,1,0,0,1,1
5,1,0,1,0,0
6,1,1,0,0,1
7,1,1,1,1,0
8,0,0,1,0,0
9,1,0,1,0,1


In [21]:
CA_generator_AETG = AETGGenerator(p, t=3)
CA3 = CA_generator_AETG.generate()
print(CA3.is_valid())
CA3.to_data_frame()

(True, [])


,0,1,2,3,4
0,0,1,1,1,0
1,0,0,1,0,1
2,1,0,0,1,1
3,0,0,0,0,0
4,1,1,0,0,1
5,1,0,1,0,0
6,0,1,0,1,1
7,1,1,1,1,1
8,1,1,0,1,0
9,0,1,1,0,0


In [22]:
def benchmark(ks: list[int], vs: list[int], Generators: CoveringArrayGenerator) -> dict[CoveringArrayGenerator.__name__, dict[tuple[int, int], dict['time' or 'n', float or int]]]:
    res = {
        Generator.__name__: {
            (k, v): { "time": float('inf'), "n": None }
            for k, v in itertooCPSATGeneratorls.product(ks, vs)
        }
        for Generator in Generators
    }

    nb_test_to_do = len(ks) * len(vs) * len(Generators)
    test_number = 1
    
    for k in ks:
        for v in vs:
            iter_param = gen_benchmark_parameters(k, v)
            for Generator in Generators:
                print(f"Test n°{test_number}/{nb_test_to_do}: ({k=}, {v=}), Generator={Generator.__name__.replace("Generator", "")} - ", end = "")
                test_number += 1
                if k >= 6 and v >= 4 and Generator == CPSATGenerator:
                    print("SKIPPING (OOM)")
                    continue

                instance = Generator(iter_param, 3)
                failed = False
                start = perf_counter()
                try:
                    CA = instance.generate()
                except:
                    failed = True
                end = perf_counter()

                if failed:
                    print(f"Failed to resolve in a timely manner or got an error")
                    res[Generator.__name__][k, v]["time"] = float('inf')
                    res[Generator.__name__][k, v]["n"] = None
                else:
                    t = end - start
                    n = len(CA.get_matrix())
                    print(f"Got {n = } in {t = }")
                    res[Generator.__name__][k, v]["time"] = t
                    res[Generator.__name__][k, v]["n"] = n
    return res

In [16]:
benchmark_results = benchmark([4, 5, 6], [2, 3, 4], [CPSATGenerator, IPOGGenerator, AETGGenerator])

Test n°1/27: (k=4, v=2), Generator=CPSAT - Got n = 8 in t = 0.24505256799966446
Test n°2/27: (k=4, v=2), Generator=IPOG - Got n = 8 in t = 0.0003370710001036059
Test n°3/27: (k=4, v=2), Generator=AETG - Got n = 8 in t = 0.8032895480027946
Test n°4/27: (k=4, v=3), Generator=CPSAT - Got n = 27 in t = 5.972962280000502
Test n°5/27: (k=4, v=3), Generator=IPOG - Got n = 33 in t = 0.0033342070018989034
Test n°6/27: (k=4, v=3), Generator=AETG - Got n = 29 in t = 5.8534634730021935
Test n°7/27: (k=4, v=4), Generator=CPSAT - Got n = 64 in t = 185.77514299500035
Test n°8/27: (k=4, v=4), Generator=IPOG - Got n = 64 in t = 0.017431166997994296
Test n°9/27: (k=4, v=4), Generator=AETG - Got n = 74 in t = 28.27297235700098
Test n°10/27: (k=5, v=2), Generator=CPSAT - Got n = 10 in t = 24.887664616999245
Test n°11/27: (k=5, v=2), Generator=IPOG - Got n = 15 in t = 0.0008728829998290166
Test n°12/27: (k=5, v=2), Generator=AETG - Got n = 11 in t = 1.686003893999441
Test n°13/27: (k=5, v=3), Generator=CPS

In [19]:
benchmark_results

{'CPSATGenerator': {(4, 2): {'time': 0.24505256799966446, 'n': 8},
  (4, 3): {'time': 5.972962280000502, 'n': 27},
  (4, 4): {'time': 185.77514299500035, 'n': 64},
  (5, 2): {'time': 24.887664616999245, 'n': 10},
  (5, 3): {'time': 185.79601469300178, 'n': 36},
  (5, 4): {'time': 232.21063664800022, 'n': 625},
  (6, 2): {'time': 181.62785836099647, 'n': 12},
  (6, 3): {'time': 225.2544711699993, 'n': 44},
  (6, 4): {'time': inf, 'n': None}},
 'IPOGGenerator': {(4, 2): {'time': 0.0003370710001036059, 'n': 8},
  (4, 3): {'time': 0.0033342070018989034, 'n': 33},
  (4, 4): {'time': 0.017431166997994296, 'n': 64},
  (5, 2): {'time': 0.0008728829998290166, 'n': 15},
  (5, 3): {'time': 0.010829673999978695, 'n': 47},
  (5, 4): {'time': 0.05079345400008606, 'n': 64},
  (6, 2): {'time': 0.002362143000937067, 'n': 17},
  (6, 3): {'time': 0.026294011000572937, 'n': 60},
  (6, 4): {'time': 0.17693142099960824, 'n': 64}},
 'AETGGenerator': {(4, 2): {'time': 0.8032895480027946, 'n': 8},
  (4, 3): {'